In [ ]:
!pip install pandas numpy matplotlib seaborn tqdm scikit-learn datasets
!pip install emoji
!pip uninstall nltk -y
!pip install nltk
!pip install spacy
!python -m spacy download en_core_web_sm


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 487.4/487.4 kB 5.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 116.3/116.3 kB 6.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 143.5/143.5 kB 8.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 194.8/194.8 kB 9.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 590.6/590.6 kB 7.7 MB/s eta 0:00:00
Found existing installation: nltk 3.9.1
Uninstalling nltk-3.9.1:
  Successfully uninstalled nltk-3.9.1
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.5/1.5 MB 16.3 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.8/12.8 MB 51.5 MB/s eta 0:00:00
✔ Download and installation successful
You can now load the package via spacy.load('en_core_web_sm')
⚠ Restart to reload dependencies
If you are in a Jupyter or Colab notebook, you may need to restart Python in
order to load all the package's dependencies. You can do this by selecting the
'Restart kernel' or 'Restart runtime' option.


In [ ]:
# 📌 File and Directory Management
import os
import tarfile

# 📌 Data Manipulation
import pandas as pd
import numpy as np

# 📌 Data Visualization
import matplotlib.pyplot as plt
import seaborn as sns
from wordcloud import WordCloud

# 📌 Text Processing (NLP)
import re
import emoji
import nltk
from nltk.corpus import stopwords
from textblob import TextBlob
from collections import Counter
import itertools
from nltk.tokenize import word_tokenize

# 📌 Machine Learning and Utilities
from tqdm import tqdm
from sklearn.model_selection import train_test_split
from sklearn.utils import shuffle

# 📌 Pandas Display Settings
pd.set_option("display.max_colwidth", None)  # Show entire text columns


In [ ]:
# NLTK
nltk.download('punkt')
nltk.download('stopwords')
stop_words = set(stopwords.words('english'))

[nltk_data] Downloading package punkt to /root/nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


# Open zipped file

# Processing

In [17]:
# File path for the tar archive
tar_path = "aclImdb_v1.tar"

# Extracting the files
with tarfile.open(tar_path, "r") as tar:
    tar.extractall()  # Extracts to the current directory

print("File extracted successfully!")


File extracted successfully!


In [20]:
# Dataset path
dataset_path = "aclImdb"

# Function to load data
def load_imdb_data(split):
    data = []
    labels = []
    for sentiment, label in [("pos", 1), ("neg", 0)]:
        path = os.path.join(dataset_path, split, sentiment)
        for filename in tqdm(os.listdir(path), desc=f"Loading {split}/{sentiment}"):
            with open(os.path.join(path, filename), "r", encoding="utf-8") as file:
                data.append(file.read())
                labels.append(label)
    return pd.DataFrame({"review": data, "sentiment": labels})

# Loading
df_train_bert = load_imdb_data("train")
df_test_bert = load_imdb_data("test")

Loading test/neg: 100%|██████████| 12500/12500 [00:00<00:00, 23703.31it/s]


In [22]:
# Light cleanup function for BERT
def clean_text_bert(text):
    text = re.sub(r"<.*?>", "", text)  # Remove HTML
    text = re.sub(r"\s+", " ", text)   # Normalize spaces
    text = text.strip()
    return text

# Apply preprocessing
df_train_bert["clean_review"] = df_train_bert["review"].apply(clean_text_bert)
df_test_bert["clean_review"] = df_test_bert["review"].apply(clean_text_bert)

In [23]:
# Remove duplicates
df_train_bert = df_train_bert.drop_duplicates(subset=['clean_review'], keep='first')
df_test_bert = df_test_bert.drop_duplicates(subset=['clean_review'], keep='first')

# Validation

In [24]:
# Complete validation: training set
print("📌 Training Validation (BERT):")
df_train_bert['is_empty'] = df_train_bert['clean_review'].apply(lambda x: len(str(x).strip()) == 0)
df_train_bert['is_too_short'] = df_train_bert['clean_review'].apply(lambda x: len(str(x).split()) < 3)
print(f"📌 Empty reviews (train): {df_train_bert['is_empty'].sum()}")
print(f"📌 Very short reviews (< 3 words, train): {df_train_bert['is_too_short'].sum()}")
df_train_bert = df_train_bert[~df_train_bert['is_empty']]
df_train_bert['review_length'] = df_train_bert['clean_review'].apply(lambda x: len(str(x).split()))
print(f"📌 Average length of reviews (train): {df_train_bert['review_length'].mean()}")
print(f"📌 Maximum length of reviews (train): {df_train_bert['review_length'].max()}")
df_train_bert['has_html'] = df_train_bert['clean_review'].apply(lambda x: bool(re.search(r"<.*?>", str(x))))
print(f"📌 Reviews with residual HTML (train): {df_train_bert['has_html'].sum()}")
print(f"📌 Training size after removing empty reviews: {len(df_train_bert)}")

# Complete validation: test set
print("\n📌 Test Validation (BERT):")
df_test_bert['is_empty'] = df_test_bert['clean_review'].apply(lambda x: len(str(x).strip()) == 0)
df_test_bert['is_too_short'] = df_test_bert['clean_review'].apply(lambda x: len(str(x).split()) < 3)
print(f"📌 Empty reviews (test): {df_test_bert['is_empty'].sum()}")
print(f"📌 Very short reviews (< 3 words, test): {df_test_bert['is_too_short'].sum()}")
df_test_bert = df_test_bert[~df_test_bert['is_empty']]
df_test_bert['review_length'] = df_test_bert['clean_review'].apply(lambda x: len(str(x).split()))
print(f"📌 Average length of reviews (test): {df_test_bert['review_length'].mean()}")
print(f"📌 Maximum length of reviews (test): {df_test_bert['review_length'].max()}")
df_test_bert['has_html'] = df_test_bert['clean_review'].apply(lambda x: bool(re.search(r"<.*?>", str(x))))
print(f"📌 Reviews with residual HTML (test): {df_test_bert['has_html'].sum()}")
print(f"📌 Test size after removing empty reviews: {len(df_test_bert)}")

📌 Training Validation (BERT):
📌 Empty reviews (train): 0
📌 Very short reviews (< 3 words, train): 0
📌 Average length of reviews (train): 229.92510942456732
📌 Maximum length of reviews (train): 2450
📌 Reviews with residual HTML (train): 0
📌 Training size after removing empty reviews: 24903

📌 Test Validation (BERT):
📌 Empty reviews (test): 0
📌 Very short reviews (< 3 words, test): 0
📌 Average length of reviews (test): 224.70271360025805
📌 Maximum length of reviews (test): 2192
📌 Reviews with residual HTML (test): 0
📌 Test size after removing empty reviews: 24801


In [25]:
# Save processed data
df_train_bert[['clean_review', 'sentiment']].to_csv('imdb_train_bert.csv', index=False)
df_test_bert[['clean_review', 'sentiment']].to_csv('imdb_test_bert.csv', index=False)
print("📌 Data for BERT saved: 'imdb_train_bert.csv' and 'imdb_test_bert.csv'")

📌 Data for BERT saved: 'imdb_train_bert.csv' and 'imdb_test_bert.csv'
